# 02. Tensors, Views, Strides & Memory Layouts

- **Category:** 00. Foundations & Tensor Calculus
- **Lineage & Citations:** Iverson (1962) *A Programming Language (APL)*; Paszke et al. (2019) *PyTorch: An Imperative Style, High-Performance Deep Learning Library*; Rogozhnikov (2022) *Einops: Clear and Reliable Tensor Operations*.
- **Purpose:** Implements the fundamental mechanics of n-dimensional tensor storage, memory strides, zero-copy view transformations, contiguous vs non-contiguous layout dynamics, and path-optimized Einstein summation (einsum).

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> 1. Do not perform in-place writes (`.add_()`, `*=`) on tensors produced via `.expand()` or `as_strided` with overlapping strides; doing so silently corrupts multiple coordinates simultaneously.
> 2. Do not call `.view()` on non-contiguous transposed tensors; use `.reshape()` or `.contiguous()`.
> 3. Do not run unparenthesized multi-operand einsum expressions on large tensors without `optimize=True`.

## Mathematical Formulation & Complexity

### 1. Linear Storage Addressing
A multidimensional coordinate $\mathbf{i} = (i_0, i_1, \dots, i_{D-1})$ maps to a scalar 1D physical memory offset $L(\mathbf{i})$ via strides $\mathbf{s} = (s_0, \dots, s_{D-1})$:
$$L(\mathbf{i}) = \text{offset} + \sum_{k=0}^{D-1} i_k \cdot s_k$$

For Row-Major (C-contiguous) layout: $s_{D-1} = 1, \ s_k = s_{k+1} \cdot d_{k+1}$.

### 2. Zero-Copy Window Views & Invariants
A 1D signal $[B, L]$ with strides $(s_b, s_l)$ is re-strided into overlapping windows of size $K$ and step $S$ without memory copying:
$$\text{Shape: } (B, W, K), \quad \text{Strides: } (s_b, S \cdot s_l, s_l), \quad W = \lfloor(L - K)/S\rfloor + 1$$

**Asymptotic Complexity:** Stride view construction: $\mathcal{O}(1)$ time, $\mathcal{O}(1)$ memory; Naive tensor copy: $\mathcal{O}(B \cdot W \cdot K)$ time & memory.

In [ ]:
import torch

import core

# 1. Deterministic Environment Setup
env = core.setup_env(seed=42)

# 2. Procedural Multi-Domain Sequence Data
batch_size = 4
seq_len = 64
x_seq = torch.linspace(0.0, 100.0, batch_size * seq_len, dtype=torch.float32).reshape(
    batch_size, seq_len
)

print(
    f"Device: {env.device} | Tensor: shape={x_seq.shape}, stride={x_seq.stride()}, contiguous={x_seq.is_contiguous()}"
)

In [ ]:
# 1. Stride Offset Calculator
def calculate_linear_offset(tensor: torch.Tensor, coords: tuple[int, ...]) -> int:
    offset = tensor.storage_offset()
    for c, s in zip(coords, tensor.stride()):
        offset += c * s
    return offset


# 2. Zero-Copy Sliding Window Extractor via as_strided
def extract_sliding_windows_zero_copy(
    x: torch.Tensor, window_size: int, step: int = 1
) -> torch.Tensor:
    batch_size, length = x.shape
    num_windows = (length - window_size) // step + 1
    b_stride, l_stride = x.stride()
    new_shape = (batch_size, num_windows, window_size)
    new_strides = (b_stride, step * l_stride, l_stride)
    return torch.as_strided(x, size=new_shape, stride=new_strides)


# 3. Path-Optimized Einsum Contraction
def batched_bilinear_einsum(x: torch.Tensor, w: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bi,ij,bj->b", x, w, y)


windows = extract_sliding_windows_zero_copy(x_seq, window_size=8, step=4)
print(
    f"Sliding Windows Extracted: shape={windows.shape}, storage_shared={x_seq.data_ptr() == windows.data_ptr()}"
)

In [ ]:
# Quantitative Verification & Stride Invariant Checks
# 1. Stride Arithmetic Validation across arbitrary coordinates
t_3d = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
for i in range(2):
    for j in range(3):
        for k in range(4):
            calc_idx = calculate_linear_offset(t_3d, (i, j, k))
            storage_val = t_3d.untyped_storage()[calc_idx * 4 : (calc_idx + 1) * 4]
            assert t_3d[i, j, k].item() == t_3d.view(-1)[calc_idx].item()

# 2. Zero-Copy Invariant Assertion (Zero byte allocations)
assert x_seq.data_ptr() == windows.data_ptr(), (
    "Window view must share exact storage pointer with source tensor!"
)

# 3. Einsum Contraction Equivalence to GEMM
b, d_in, d_out = 8, 16, 32
x_in = torch.randn(b, d_in)
w_mat = torch.randn(d_in, d_out)
y_in = torch.randn(b, d_out)
res_einsum = batched_bilinear_einsum(x_in, w_mat, y_in)
res_gemm = (x_in @ w_mat * y_in).sum(dim=-1)
torch.testing.assert_close(res_einsum, res_gemm, atol=1e-5, rtol=1e-5)
print("All stride arithmetic and einsum invariants passed successfully.")

In [ ]:
# Standardized Performance Benchmark Table
from core.benchmark import benchmark_fn, format_benchmark_table


# Compare zero-copy as_strided vs naive clone unfold
def naive_unfold(x: torch.Tensor, k: int, s: int) -> torch.Tensor:
    num_w = (x.shape[1] - k) // s + 1
    return torch.stack([x[:, i * s : i * s + k].clone() for i in range(num_w)], dim=1)


b_zero_copy = benchmark_fn(
    lambda: extract_sliding_windows_zero_copy(x_seq, 8, 4), iters=50, name="Zero-Copy as_strided"
)
b_naive_unfold = benchmark_fn(
    lambda: naive_unfold(x_seq, 8, 4), iters=50, name="Naive Clone Unfold"
)
b_einsum = benchmark_fn(
    lambda: batched_bilinear_einsum(x_in, w_mat, y_in), iters=50, name="Optimized Einsum"
)

print(format_benchmark_table([b_zero_copy, b_naive_unfold, b_einsum]))

In [ ]:
# Runnable Misuse Contrast: Overlapping Strides Mutation vs Cloned Mutation
# FAILURE MODE: Mutating an overlapping strided view corrupts shared memory addresses!
base_tensor = torch.zeros(4, dtype=torch.float32)

# 1. Broadcast View with Stride 0 (Overlapping Memory)
overlap_view = torch.as_strided(base_tensor, size=(4,), stride=(0,))
overlap_view[0] += 5.0  # Attempt to mutate only coordinate 0

corrupted_values = base_tensor.clone()
print(f"[FAILURE] Mutating overlap view corrupted underlying buffer: {corrupted_values.tolist()}")
# All elements in the broadcast view are now 5.0 because they share address 0!
assert corrupted_values[0].item() == 5.0 and overlap_view[3].item() == 5.0

# 2. Safe Cloned Mutation (Independent Memory)
safe_tensor = torch.zeros(4, dtype=torch.float32)
safe_copy = safe_tensor.clone()
safe_copy[0] += 5.0
print(
    f"[CORRECT] Safe cloned mutation preserves source: source={safe_tensor.tolist()}, mutated={safe_copy.tolist()}"
)
assert safe_tensor[0].item() == 0.0 and safe_copy[0].item() == 5.0

## Architectural Layout, Key Takeaways & Relative Navigation

### Memory Storage & Stride Topology
<svg width="650" height="150" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <text x="30" y="30" fill="#94a3b8" font-size="12" font-weight="bold">PHYSICAL STORAGE vs MULTI-DIMENSIONAL STRIDED VIEWS</text>
  <!-- 1D Physical Storage Buffer -->
  <rect x="30" y="45" width="590" height="30" rx="4" fill="#1e293b" stroke="#64748b" stroke-width="1.5"/>
  <text x="40" y="65" fill="#f8fafc" font-size="11">Linear Storage: [ byte 0 ][ byte 4 ][ byte 8 ][ byte 12 ][ byte 16 ][ byte 20 ][ byte 24 ][ byte 28 ]</text>
  <!-- 2D View Mapping -->
  <rect x="30" y="95" width="280" height="40" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="170" y="115" fill="#38bdf8" font-size="11" text-anchor="middle">Contiguous View: Shape (2, 4), Stride (4, 1)</text>
  <text x="170" y="128" fill="#94a3b8" font-size="10" text-anchor="middle">L(i, j) = i * 4 + j * 1 (Linear Step)</text>
  <!-- Transposed View -->
  <rect x="340" y="95" width="280" height="40" rx="6" fill="#1e293b" stroke="#f43f5e" stroke-width="1.5"/>
  <text x="480" y="115" fill="#f43f5e" font-size="11" text-anchor="middle">Transposed View: Shape (4, 2), Stride (1, 4)</text>
  <text x="480" y="128" fill="#94a3b8" font-size="10" text-anchor="middle">L(i, j) = i * 1 + j * 4 (Non-Contiguous!)</text>
</svg>

### Key Takeaways
1. **Zero-Copy Views:** Operations like `as_strided`, `transpose`, `permute`, and slicing do not copy data; they merely instantiate new tensor headers with recalculated shape and stride tuples.
2. **Cache Coalescing:** Memory access along stride-1 dimensions achieves full memory bus utilization (64-byte burst lines), whereas high strides cause severe cache miss penalties.
3. **In-Place Safety:** Never mutate a view whose strides overlap ($s_k = 0$ or overlapping window spans), as changes propagate across multiple logical indices simultaneously.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_tensors_views_strides_and_memory_layouts.md](../../markdown/00_foundations/02_tensors_views_strides_and_memory_layouts.md)
- **PyTorch 2.x Production Module:** [02_tensors_views_strides_and_memory_layouts.py](../../pytorch2/00_foundations/02_tensors_views_strides_and_memory_layouts.py)
- **Functional JAX Track:** [02_tensors_views_strides_and_memory_layouts.py](../../jax/00_foundations/02_tensors_views_strides_and_memory_layouts.py)
- **Native Rust Candle Track:** [02_tensors_views_strides_and_memory_layouts.rs](../../candle/00_foundations/02_tensors_views_strides_and_memory_layouts.rs)
- **CPU Fused Kernel:** [02_tensors_views_strides_and_memory_layouts_cpu.py](../../kernels/00_foundations/02_tensors_views_strides_and_memory_layouts_cpu.py)
- **Golden Reference Checkpoint:** [02_tensors_views_strides_and_memory_layouts.safetensors](../../weights/00_foundations/02_tensors_views_strides_and_memory_layouts.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Linear Neuron & Invertible Transforms](01_linear_neuron_and_invertible_transforms.ipynb) | [Master Index: README.md](../../README.md) | [Next: 03. Autograd Mechanics & Wirtinger Calculus](03_autograd_mechanics_complex_wirtinger_and_custom_functions.ipynb) &rarr;
